# Solar Potential Energy vs. Solar Energy Production U.S. Cartogram

## Steps

- Data Preparation
  - Raster Data: Extract average DNI/GHI per state using the shapefile
  - Solar Energy Data: Aggregate solar production by state using the shapefile
- Metric Calculation
  - solar_efficiency = production / resource
- Cartogram Generation
  - use cartogram-geopandas, scapetoad, or geocart
- Visualization
  - matplotlib, folium, or geopandas

- Packages
  - rasterstats – To extract raster values by polygons (states).
  - geopandas – For handling vector data.
  - rasterio – For reading raster data.
  - cartogram_geopandas – For producing cartograms in Python.
  - matplotlib – For final visualization.

## Concept: GHI vs. DNI

${GHI} = \text{DHI} + \text{DNI} \cdot \cos(\theta)$

- **GHI**: Global Horizontal Irradiance – total solar radiation received per unit area on a horizontal surface.
- **DHI**: Diffuse Horizontal Irradiance – scattered sunlight that reaches the surface from all directions (excluding the direct sun).
- **DNI**: Direct Normal Irradiance – direct sunlight received per unit area, measured on a surface perpendicular to the sun’s rays.
- **θ (theta)**: Solar zenith angle – the angle between the sun and the vertical direction (0° when the sun is directly overhead, 90° when on the horizon).

Why the difference matters

- **PV panels** are typically tilted, so using just horizontal irradiance may not reflect actual power generation.
- Knowing **both DHI and DNI** enables accurate modeling of solar panel performance and overall solar resource estimation.

**Source**: [The Difference between Horizontal and Tilted Global Solar Irradiance](https://www.kippzonen.com/News/408/The-Difference-between-Horizontal-and-Tilted-Global-Solar-Irradiance) – Kipp & Zonen


In [ ]:
%pip install rasterstats
%pip install us
%pip install geoplot
%pip install mapclassify
%pip install contextily

In [ ]:
# Import Libraries
import rasterio
import numpy as np
import pandas as pd
import plotly.express as px
import rasterio
from rasterio.merge import merge
from rasterio.plot import show
from rasterstats import zonal_stats
import geopandas as gpd
import matplotlib.pyplot as plt
import geoplot as gplt
import tempfile
import os
import folium
from folium.features import GeoJsonTooltip
import us 
import branca.colormap as cm
import us  # For mapping state abbreviations to full names (near end of notebook)
from shapely.affinity import scale
from sklearn.preprocessing import MinMaxScaler

# Data Preparation

Investigating the 'Annual DNI' shapefile

In [ ]:
# load raster data
tiff = rasterio.open(r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Annual DNI\nsrdb3_dni.tif')
data = tiff.read(1)
data[data == tiff.nodata] = np.nan

# Create interactive plot
fig = px.imshow(
    data,
    color_continuous_scale='YlOrRd',
    origin='upper',
    title='Zoomable DNI Raster',
    labels={'color': 'Wh/m²/day'}
)

# Maximize layout
fig.update_layout(
    autosize=True,
    width=None,
    height=None,
    margin=dict(l=0, r=0, t=30, b=0),  # keep small top margin for title
    coloraxis_colorbar=dict(title='DNI'),
    dragmode='pan'
)

fig.show()


### Raster Data

#### DNI (Direct Normal Irradiance)

In [ ]:
# load state boundaries
states = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\tl_2024_us_state\tl_2024_us_state.shp"
)

# load raster to get CRS
RASTER_PATH_DNI = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Annual DNI\nsrdb3_dni.tif"
with rasterio.open(RASTER_PATH_DNI) as src:
    raster_crs = src.crs

# reproject state geometries to match raster for zonal stats
states_zonal = states.to_crs(raster_crs)

# run zonal stats (normalized values using mean)
zstats = zonal_stats(states_zonal, RASTER_PATH_DNI, stats=["mean"], geojson_out=False, all_touched=True)

# attach mean DNI to original state geometries
states = states.copy()
states["dni_per_m2"] = [stat["mean"] for stat in zstats]  # Mean DNI per square meter

# Convert to WGS84 for folium
dni_gdf = states.to_crs(epsg=4326)

# build folium map
m = folium.Map(location=[39.8283, -98.5795], zoom_start=5)

folium.Choropleth(
    geo_data=dni_gdf,
    data=dni_gdf,
    columns=["NAME", "dni_per_m2"],
    key_on="feature.properties.NAME",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Mean Daily DNI per m² (Wh/m²/day)"
).add_to(m)

tooltip = GeoJsonTooltip(
    fields=["NAME", "dni_per_m2"],
    aliases=["State:", "Mean DNI (Wh/m²/day):"],
    localize=True,
    sticky=True
)

folium.GeoJson(
    dni_gdf,
    tooltip=tooltip,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0
    }
).add_to(m)

# add title
map_title = "Mean daily DNI per m^2"
title_html = f'<h1 style="position:absolute;z-index:100000;left:10vw" >{map_title}</h1>'
m.get_root().html.add_child(folium.Element(title_html))

m

save image

In [ ]:
# m.save(r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\images\mean_daily_DNI.html")

#### GHI (Global Horizontal Irradiance)

In [ ]:
# load state boundaries
states = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\tl_2024_us_state\tl_2024_us_state.shp"
)

# load GHI raster to get CRS
RASTER_PATH_GHI = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_ghi\nsrdbv3_ghi\Annual GHI\nsrdb3_ghi.tif"
with rasterio.open(RASTER_PATH_GHI) as src:
    raster_crs = src.crs

# reproject state geometries to match raster for zonal stats
states_zonal = states.to_crs(raster_crs)

# run zonal stats (normalized values using mean GHI)
zstats = zonal_stats(states_zonal, RASTER_PATH_GHI, stats=["mean"], geojson_out=False, all_touched=True)

# attach mean GHI to original state geometries
states = states.copy()
states["ghi_per_m2"] = [stat["mean"] for stat in zstats]  # Mean GHI per square meter

# convert to WGS84 for folium
ghi_gdf = states.to_crs(epsg=4326)

# build folium map
m = folium.Map(location=[39.8283, -98.5795], zoom_start=5)

folium.Choropleth(
    geo_data=ghi_gdf,
    data=ghi_gdf,
    columns=["NAME", "ghi_per_m2"],
    key_on="feature.properties.NAME",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Mean Daily GHI per m² (Wh/m²/day)"
).add_to(m)

tooltip = GeoJsonTooltip(
    fields=["NAME", "ghi_per_m2"],
    aliases=["State:", "Mean GHI (Wh/m²/day):"],
    localize=True,
    sticky=True
)

folium.GeoJson(
    ghi_gdf,
    tooltip=tooltip,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0
    }
).add_to(m)

# add title
map_title = "Mean Daily GHI per m^2"
title_html = f'<h1 style="position:absolute;z-index:100000;left:10vw" >{map_title}</h1>'
m.get_root().html.add_child(folium.Element(title_html))

m


save image

In [ ]:
# m.save(r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\images\mean_daily_GHI.html")

### Solar Energy Data

In [ ]:
energy_production = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\PowerPlants\PowerPlants_US_EIA.shp"
)

# previuew columns & header
print(energy_production.columns)
print(energy_production.head())

filter for non-zero values

In [ ]:
# filter to only plants with NON-ZERO solar capacity 
solar_production = energy_production[energy_production["Solar_MW"] > 0]

# inspect the result 
print(solar_production.shape)
print(solar_production[["Plant_Name", "State", "Solar_MW"]].head())

summarize by state

In [ ]:
# group by state and sum the solar MW 
solar_by_state = solar_production.groupby("State")["Solar_MW"].sum().reset_index()

# Sort descending
solar_by_state = solar_by_state.sort_values(by="Solar_MW", ascending=False)

print(solar_by_state.head(10))

#### Create rudimentary map ~ using the Power Plants dataset

In [ ]:
# load summarized solar capacity per state 
solar_by_state = solar_production.groupby("State")[["Solar_MW"]].sum().reset_index()

states = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\tl_2024_us_state\tl_2024_us_state.shp"
)

# clean up for merging 
solar_by_state["State"] = solar_by_state["State"].str.upper()
states["NAME"] = states["NAME"].str.upper()

# merge the solar capacity data into the states GeoDataFrame 
states = states.merge(solar_by_state, left_on="NAME", right_on="State", how="left")

# Fill NaNs for states without solar with 0 
states["Solar_MW"] = states["Solar_MW"].fillna(0)

# reproject to WGS84 for folium 
solar_gdf = states.to_crs(epsg=4326)

# build map 
m = folium.Map(location=[39.8283, -98.5795], zoom_start=5)

folium.Choropleth(
    geo_data=solar_gdf,
    data=solar_gdf,
    columns=["NAME", "Solar_MW"],
    key_on="feature.properties.NAME",
    fill_color="YlGnBu",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Installed Solar Capacity (MW)"
).add_to(m)

tooltip = GeoJsonTooltip(
    fields=["NAME", "Solar_MW"],
    aliases=["State:", "Installed Solar Capacity (MW):"],
    localize=True,
    sticky=True
)

folium.GeoJson(
    solar_gdf,
    tooltip=tooltip,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0
    }
).add_to(m)

In [ ]:
high_power_solar = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\HPspSHP\uspvdb_v2_0_20240801.shp"
)

# reproject to WGS84 for folium 
high_power_solar = high_power_solar.to_crs(epsg=4326)

# setup default style function 
style_fn = lambda feature: {
    'fillColor': 'orange',
    'color': 'red',
    'weight': 1.5,
    'fillOpacity': 0.5
}

# conditionally add tooltip if fields exist 
if "STATE" in high_power_solar.columns and "NAME" in high_power_solar.columns:
    tooltip = GeoJsonTooltip(
        fields=["STATE", "NAME"],
        aliases=["State:", "Site:"],
        sticky=True
    )
    folium.GeoJson(
        high_power_solar,
        name="High-Powered Solar Installations",
        style_function=style_fn,
        tooltip=tooltip
    ).add_to(m)
else:
    folium.GeoJson(
        high_power_solar,
        name="High-Powered Solar Installations",
        style_function=style_fn
    ).add_to(m)

# add title
map_title = "High power solar panel generation amounts & locations"
title_html = f'<h1 style="position:absolute;z-index:100000;left:10vw" >{map_title}</h1>'
m.get_root().html.add_child(folium.Element(title_html))


# add layer control 
folium.LayerControl().add_to(m)

m


#### Create rudimentary map ~ Using the High Powered Solar Panels Dataset

In [ ]:
solar_sites = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\HPspSHP\uspvdb_v2_0_20240801.shp"
)

# drop missing values in capacity or state
solar_sites_clean = solar_sites.dropna(subset=["p_state", "p_cap_ac"])

# group by state and sum the AC capacity
solar_by_state = solar_sites_clean.groupby("p_state")["p_cap_ac"].sum().reset_index()

# rename columns
solar_by_state.columns = ["State", "Total_Solar_Capacity_MW"]

# sort descending
solar_by_state = solar_by_state.sort_values(by="Total_Solar_Capacity_MW", ascending=False)

print(solar_by_state)

In [ ]:
# load state-level solar capacity from high-powered solar site data 
solar_by_state = solar_sites_clean.groupby("p_state")["p_cap_ac"].sum().reset_index()
solar_by_state.columns = ["State", "Total_Solar_Capacity_MW"]

# convert state abbreviations to full names 
abbr_to_name = {state.abbr: state.name.upper() for state in us.states.STATES}
solar_by_state["State"] = solar_by_state["State"].map(abbr_to_name)

states = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\tl_2024_us_state\tl_2024_us_state.shp"
)

# clean up for merging 
states["NAME"] = states["NAME"].str.upper()

# merge the solar capacity data into the states GeoDataFrame 
states = states.merge(solar_by_state, left_on="NAME", right_on="State", how="left")

# fill NaNs for states without solar with 0 
states["Total_Solar_Capacity_MW"] = states["Total_Solar_Capacity_MW"].fillna(0)

# reproject to WGS84 for folium 
solar_gdf = states.to_crs(epsg=4326)

# Build map 
m = folium.Map(location=[39.8283, -98.5795], zoom_start=5)

folium.Choropleth(
    geo_data=solar_gdf,
    data=solar_gdf,
    columns=["NAME", "Total_Solar_Capacity_MW"],
    key_on="feature.properties.NAME",
    fill_color="YlOrBr",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Installed Solar Capacity (MW) from HPspSHP"
).add_to(m)

tooltip = GeoJsonTooltip(
    fields=["NAME", "Total_Solar_Capacity_MW"],
    aliases=["State:", "Installed HP Solar Capacity (MW):"],
    localize=True,
    sticky=True
)

folium.GeoJson(
    solar_gdf,
    tooltip=tooltip,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0
    }
).add_to(m)

In [ ]:
high_power_solar = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\HPspSHP\uspvdb_v2_0_20240801.shp"
)

# reproject to WGS84 
high_power_solar = high_power_solar.to_crs(epsg=4326)

# add high-powered solar polygons 
folium.GeoJson(
    high_power_solar,
    name="High-Powered Solar Installations",
    style_function=lambda feature: {
        'fillColor': 'orange',
        'color': 'red',
        'weight': 1.5,
        'fillOpacity': 0.5
    },
    tooltip=GeoJsonTooltip(
        fields=["p_state", "p_name"],
        aliases=["State:", "Site Name:"],
        sticky=True
    )
).add_to(m)

# add title
map_title = "High power solar panel generation amounts & locations"
title_html = f'<h1 style="position:absolute;z-index:100000;left:10vw" >{map_title}</h1>'
m.get_root().html.add_child(folium.Element(title_html))

# add layer control 
folium.LayerControl().add_to(m)

# show map 
m

save images

In [ ]:
# m.save(r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\images\solar_panel_generation_locations.html")

# Metric Calculation

### The Formula

We are interested in comparing the fields of Solar Irradiation Resources and Current MWh of Solar Power being produced for each state.

$Estimated PV Output MWh = {(GHI × \frac{365}{1000}) * Area * Efficiency}$
- Efficiency ~ We estimate the efficiency to be around 20% per literature
- Area ~ We will sum all the values per an area on the raster file

$Utilization\ Ratio = \frac{Actual\ Solar\ Production}{Estimated\ PV\ Output}$

>1 → The state may be importing energy or has very efficient use of limited GHI

~1 → The state is efficiently using its solar resource

<1 → There's untapped solar potential

### Computing the metrics and processing

In [ ]:
abbr_to_name = {
    "AL": "ALABAMA", "AK": "ALASKA", "AZ": "ARIZONA", "AR": "ARKANSAS", "CA": "CALIFORNIA",
    "CO": "COLORADO", "CT": "CONNECTICUT", "DE": "DELAWARE", "FL": "FLORIDA", "GA": "GEORGIA",
    "HI": "HAWAII", "ID": "IDAHO", "IL": "ILLINOIS", "IN": "INDIANA", "IA": "IOWA",
    "KS": "KANSAS", "KY": "KENTUCKY", "LA": "LOUISIANA", "ME": "MAINE", "MD": "MARYLAND",
    "MA": "MASSACHUSETTS", "MI": "MICHIGAN", "MN": "MINNESOTA", "MS": "MISSISSIPPI", "MO": "MISSOURI",
    "MT": "MONTANA", "NE": "NEBRASKA", "NV": "NEVADA", "NH": "NEW HAMPSHIRE", "NJ": "NEW JERSEY",
    "NM": "NEW MEXICO", "NY": "NEW YORK", "NC": "NORTH CAROLINA", "ND": "NORTH DAKOTA", "OH": "OHIO",
    "OK": "OKLAHOMA", "OR": "OREGON", "PA": "PENNSYLVANIA", "RI": "RHODE ISLAND", "SC": "SOUTH CAROLINA",
    "SD": "SOUTH DAKOTA", "TN": "TENNESSEE", "TX": "TEXAS", "UT": "UTAH", "VT": "VERMONT",
    "VA": "VIRGINIA", "WA": "WASHINGTON", "WV": "WEST VIRGINIA", "WI": "WISCONSIN", "WY": "WYOMING",
    "DC": "DISTRICT OF COLUMBIA"
}

states = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\tl_2024_us_state\tl_2024_us_state.shp"
)

# load solar production per state from HP solar data 
solar_by_state = solar_sites_clean.groupby("p_state")["p_cap_ac"].sum().reset_index()
solar_by_state.columns = ["State", "Total_Solar_Capacity_MW"]
solar_by_state["State"] = solar_by_state["State"].map(abbr_to_name)
solar_by_state.dropna(inplace=True)

# load GHI raster and extract CRS (Coordinate Reference Systems)
RASTER_PATH_GHI = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_ghi\nsrdbv3_ghi\Annual GHI\nsrdb3_ghi.tif"
with rasterio.open(RASTER_PATH_GHI) as src:
    raster_crs = src.crs

# reproject states to raster CRS for zonal stats 
states_zonal = states.to_crs(raster_crs)

# get mean GHI per state 
zstats = zonal_stats(states_zonal, RASTER_PATH_GHI, stats=["mean"], geojson_out=False, all_touched=True)
states["ghi_per_m2"] = [stat["mean"] for stat in zstats]

# compute accurate area in square meters 
states_area = states.to_crs(epsg=5070)
states["area_m2"] = states_area.geometry.area

# clean for merge 
states["NAME"] = states["NAME"].str.upper()
solar_by_state["State"] = solar_by_state["State"].str.upper()

# merge
states = states.merge(solar_by_state, left_on="NAME", right_on="State", how="left")
states["Total_Solar_Capacity_MW"] = states["Total_Solar_Capacity_MW"].fillna(0)

# calculate Estimated PV (Photovoltaic) Output 
efficiency = 0.20
days_per_year = 365
states["Estimated_PV_Output_MWh"] = (states["ghi_per_m2"] * (days_per_year / 1000) * states["area_m2"] * efficiency)  # MWh/year

# calculate Utilization Ratio 
states["Utilization_Ratio"] = states["Total_Solar_Capacity_MW"] / states["Estimated_PV_Output_MWh"]
states["Utilization_Ratio"] = states["Utilization_Ratio"].replace([float("inf"), float("-inf")], pd.NA).fillna(0)

# Reproject for Folium 
utilization_gdf = states.to_crs(epsg=4326)

### Building the map

In [ ]:
# Build Folium Map 
m = folium.Map(location=[39.8283, -98.5795], zoom_start=4)

folium.Choropleth(
    geo_data=utilization_gdf,
    data=utilization_gdf,
    columns=["NAME", "Utilization_Ratio"],
    key_on="feature.properties.NAME",
    fill_color="RdYlGn",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Solar Utilization Ratio"
).add_to(m)

# Format to 6 decimal places
utilization_gdf["Utilization_Ratio_str"] = utilization_gdf["Utilization_Ratio"].map(lambda x: f"{x:.6f}")

tooltip = GeoJsonTooltip(
    fields=["NAME", "Utilization_Ratio_str"],
    aliases=["State:", "Utilization Ratio:"],
    localize=True,
    sticky=True
)

folium.GeoJson(
    utilization_gdf,
    tooltip=tooltip,
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0
    }
).add_to(m)

# Display the map 
m

# Cartogram Generation

#### Testing Import & Geometry

In [ ]:
# Reproject to equal-area projection 
states_proj = states.to_crs(epsg=5070)

# Normalize Utilization Ratio 
states_proj["Utilization_Ratio"] = states_proj["Utilization_Ratio"].fillna(0)
scaler = MinMaxScaler(feature_range=(0.5, 1.5))
states_proj["scaled_ratio"] = scaler.fit_transform(states_proj[["Utilization_Ratio"]])

states_proj["geometry"] = states_proj.buffer(-10_000)  # Shrinks each state slightly

# Scale geometries manually 
scaled_geometries = []
for geom, ratio in zip(states_proj.geometry, states_proj["scaled_ratio"]):
    ratio = ratio if ratio > 0 else 0.0001
    scaled_geom = scale(geom, xfact=ratio, yfact=ratio, origin='centroid')
    scaled_geometries.append(scaled_geom)

# Replace geometry 
cartogram_gdf = states_proj.copy()
cartogram_gdf["geometry"] = scaled_geometries

# Convert to EPSG:4326 to match desired bounds 
cartogram_gdf = cartogram_gdf.to_crs(epsg=4326)

# Desired geographic bounds 
DESIRED_BOUNDS = {
    'left': -130,
    'right': -65,
    'bottom': 20,
    'top': 50
}

# plot cartogram
fig, ax = plt.subplots(figsize=(14, 10))
cartogram_gdf.plot(
    column="Utilization_Ratio",
    cmap="RdYlGn",
    linewidth=0.6,
    edgecolor="0.5",
    legend=True,
    legend_kwds={
        'label': "Utilization Ratio",
        'orientation': "vertical",
        'shrink': 0.6,
    },
    ax=ax
)

# set fixed bounds for consistent map framing
ax.set_xlim(DESIRED_BOUNDS['left'], DESIRED_BOUNDS['right'])
ax.set_ylim(DESIRED_BOUNDS['bottom'], DESIRED_BOUNDS['top'])

ax.axis("off")
ax.set_title("Cartogram of US States by Solar Utilization Ratio", fontsize=18, pad=16)

plt.subplots_adjust(left=0, right=1, top=0.95, bottom=0)
plt.show()

In [ ]:
# reproject to equal-area projection for meaningful scaling 
states_proj = states.to_crs(epsg=5070)

# normalize Utilization Ratio 
states_proj["Utilization_Ratio"] = states_proj["Utilization_Ratio"].fillna(0)
scaler = MinMaxScaler(feature_range=(0.5, 1.5))
states_proj["scaled_ratio"] = scaler.fit_transform(states_proj[["Utilization_Ratio"]])

# shrink geometries slightly to reduce overlap (before scaling) 
states_proj["geometry"] = states_proj.buffer(-10_000)

# Apply manual scaling to each state geometry 
scaled_geometries = []
for geom, ratio in zip(states_proj.geometry, states_proj["scaled_ratio"]):
    factor = ratio if ratio > 0 else 0.0001  # Prevent zero scaling
    scaled_geom = scale(geom, xfact=factor, yfact=factor, origin='centroid')
    scaled_geometries.append(scaled_geom)

# replace geometry 
cartogram_gdf = states_proj.copy()
cartogram_gdf["geometry"] = scaled_geometries

# reproject to WGS84 for Folium 
cartogram_gdf = cartogram_gdf.to_crs(epsg=4326)

# format Utilization Ratio for tooltip 
cartogram_gdf["Utilization_Ratio_str"] = cartogram_gdf["Utilization_Ratio"].map(lambda x: f"{x:.6f}")

# create color scale 
min_ratio = cartogram_gdf["Utilization_Ratio"].min()
max_ratio = cartogram_gdf["Utilization_Ratio"].max()
colormap = cm.linear.RdYlGn_11.scale(min_ratio, max_ratio)
colormap.caption = "Solar Utilization Ratio"

# build Folium map
# Create the map with an initial center and zoom (fallback values)
m = folium.Map(location=[39.5, -98.5], zoom_start=5)

map_title = "Cartogram of US States by Solar Utilization Ratio"
title_html = f'<h1 style="position:absolute;z-index:100000;left:10vw" >{map_title}</h1>'
m.get_root().html.add_child(folium.Element(title_html))

# add GeoJson with dynamic coloring 
folium.GeoJson(
    cartogram_gdf,
    name="Cartogram",
    style_function=lambda feature: {
        'fillColor': colormap(feature["properties"]["Utilization_Ratio"]),
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.75,
    },
    tooltip=GeoJsonTooltip(
        fields=["NAME", "Utilization_Ratio_str"],
        aliases=["State:", "Utilization Ratio:"],
        localize=True,
        sticky=True
    )
).add_to(m)

# add color legend 
colormap.add_to(m)

# add layer control 
folium.LayerControl().add_to(m)

# Show map 
m

save cartogram

In [ ]:
# m.save(r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\images\cartogram.html")